# 5 · Deploy your own dashboards

Two Streamlit apps, straight from the workshop Git repository into **your own
schema**. No file uploads and no stage: Snowflake reads the code from Git.

| App | Runs as | Shows |
| --- | --- | --- |
| Internal MI | the app owner | the whole market |
| Partner Insights | **the viewer** | only what that viewer's role may see |

The second one is the interesting one. It contains **no filtering logic at all**.
The row access policy from notebook 04 decides what exists, so a partner user
sees one insurer and you see all seven — from identical code.

Both run on container runtime, because a Cortex Agent cannot be called from a
warehouse-runtime app.

**Edit the cell below, then run the cells in order.**

In [ ]:
-- >>> EDIT THESE TWO <<<
SET alias      = 'kkothe';         -- same alias you used in notebook 01
SET my_insurer = 'ZIXTY';          -- ZIXTY | VEYGO | COVERTIME | SAFELYINSURED

-- Rebuilt here because session variables do not carry over between notebooks.
SET my_schema = (SELECT 'DEFAQTO_DB.TRANSFORMED_' || UPPER($alias));
SET my_role   = (SELECT 'PARTNER_' || REPLACE(UPPER($my_insurer), ' ', '_'));

USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;
USE SCHEMA IDENTIFIER($my_schema);

SELECT $my_schema AS building_in, $my_role AS partner_role;

## 1 · The internal dashboard

Ordinary SQL. Two things to notice:

- **The name is unqualified**, so it is created in the schema set above. Your
  copy and everyone else's are different objects because the schemas differ.
- **`FROM` points at Git.** The path is the same for everyone; only the schema
  changes.

`ADD LIVE VERSION FROM LAST` is not optional. Without it the app runs for you
and for nobody else.

In [ ]:
CREATE OR REPLACE STREAMLIT DEFAQTO_INTERNAL_MI
  FROM '@DEFAQTO_DB.PUBLIC.WORKSHOP_REPO/branches/main/streamlit/internal_mi/'
  MAIN_FILE             = 'streamlit_app.py'
  QUERY_WAREHOUSE       = COMPUTE_WH
  RUNTIME_NAME          = 'SYSTEM$ST_CONTAINER_RUNTIME_PY3_11'
  COMPUTE_POOL          = DEFAQTO_HOL_POOL
  ARTIFACT_REPOSITORIES = (snowflake.snowpark.pypi_shared_repository)
  TITLE                 = 'Defaqto Internal MI'
  COMMENT               = 'Whole-market view. Runs as the app owner.';

ALTER STREAMLIT DEFAQTO_INTERNAL_MI ADD LIVE VERSION FROM LAST;

SHOW STREAMLITS IN SCHEMA IDENTIFIER($my_schema);

## 2 · The partner dashboard

Same shape. The difference is inside the app: it connects with
`st.connection("snowflake-callers-rights")`, so queries run as **the viewer**
rather than as the owner.

Under owner's rights the policy would evaluate `ACCOUNTADMIN`, which it exempts,
and a partner would see all seven insurers. The app would look like it worked
and be completely wrong.

In [ ]:
CREATE OR REPLACE STREAMLIT DEFAQTO_PARTNER_INSIGHTS
  FROM '@DEFAQTO_DB.PUBLIC.WORKSHOP_REPO/branches/main/streamlit/partner_insights/'
  MAIN_FILE             = 'streamlit_app.py'
  QUERY_WAREHOUSE       = COMPUTE_WH
  RUNTIME_NAME          = 'SYSTEM$ST_CONTAINER_RUNTIME_PY3_11'
  COMPUTE_POOL          = DEFAQTO_HOL_POOL
  ARTIFACT_REPOSITORIES = (snowflake.snowpark.pypi_shared_repository)
  TITLE                 = 'Defaqto Partner Insights'
  COMMENT               = 'Partner view. Runs as the viewer - the policy decides what exists.';

ALTER STREAMLIT DEFAQTO_PARTNER_INSIGHTS ADD LIVE VERSION FROM LAST;

## 3 · Let the app reach your tables

Caller's rights needs permission from **both** directions, and this is the half
people miss.

The viewer's role needs `SELECT` — your partner role got that in notebook 04.
Separately, the role that **owns the app** needs a *caller* privilege on each
object, or Snowflake refuses to reach through at all:

> `This executable runs with restricted caller's rights. The owner role
> SYSTEM$MANAGED must have at least one CALLER privilege granted on TABLE …`

The full table names are built into variables first because `IDENTIFIER()`
accepts a plain session variable but **not** a concatenated expression inside a
`GRANT` — that fails with `syntax error … unexpected '||'`.

In [ ]:
SET cg_prov = $my_schema || '.GOLD_PROVIDER_DAILY';
SET cg_coh  = $my_schema || '.GOLD_COHORT_CONVERSION';
SET cg_fun  = $my_schema || '.GOLD_FUNNEL_DAILY';

GRANT CALLER USAGE  ON DATABASE DEFAQTO_DB             TO ROLE ACCOUNTADMIN;
GRANT CALLER USAGE  ON SCHEMA   IDENTIFIER($my_schema) TO ROLE ACCOUNTADMIN;

GRANT CALLER SELECT ON DYNAMIC TABLE IDENTIFIER($cg_prov) TO ROLE ACCOUNTADMIN;
GRANT CALLER SELECT ON DYNAMIC TABLE IDENTIFIER($cg_coh)  TO ROLE ACCOUNTADMIN;
GRANT CALLER SELECT ON DYNAMIC TABLE IDENTIFIER($cg_fun)  TO ROLE ACCOUNTADMIN;

SHOW CALLER GRANTS TO ROLE ACCOUNTADMIN;

## 4 · Let your partner open it

A container-runtime app needs **three** grants per viewer role, not one: the
Streamlit, the service running it, and that service's `STREAMLIT_VIEWER` role.

Service names are generated, so run the `SHOW SERVICES` first and paste the name
that matches your app into the two lines below.

In [ ]:
GRANT USAGE ON STREAMLIT DEFAQTO_PARTNER_INSIGHTS TO ROLE IDENTIFIER($my_role);

-- Find the service behind your app:
SHOW SERVICES IN COMPUTE POOL DEFAQTO_HOL_POOL;

-- Then substitute its name into these two and run them:
--
--   GRANT USAGE ON SERVICE      <db>.<schema>.<service>                  TO ROLE PARTNER_ZIXTY;
--   GRANT USAGE ON SERVICE ROLE <db>.<schema>.<service>.STREAMLIT_VIEWER TO ROLE PARTNER_ZIXTY;

SHOW GRANTS TO ROLE IDENTIFIER($my_role);

## 5 · Prove the isolation

Same query, twice, as two different roles. Nothing about the query changes.

| Acting as | Insurers visible |
| --- | --- |
| your partner role | **1** |
| `ACCOUNTADMIN` | 7 |

In [ ]:
USE ROLE IDENTIFIER($my_role);
SELECT CURRENT_ROLE()               AS acting_as,
       COUNT(DISTINCT PROVIDER_KEY) AS insurers_visible,
       COUNT(*)                     AS rows_visible
FROM   IDENTIFIER($cg_prov);

USE ROLE ACCOUNTADMIN;
SELECT CURRENT_ROLE()               AS acting_as,
       COUNT(DISTINCT PROVIDER_KEY) AS insurers_visible,
       COUNT(*)                     AS rows_visible
FROM   IDENTIFIER($cg_prov);

## 6 · Open them

**Projects » Streamlit.** Both apps are titled the same for everyone, so find
yours by schema — `TRANSFORMED_<your alias>`.

Then sign in as your partner user and open the partner app. One insurer, not
seven.

**If the partner still sees seven**, work through these in order — each has
caused it at least once:

1. **The partner user's DEFAULT role.** Caller's rights uses the *default* role,
   not the one selected in Snowsight. If that is `ACCOUNTADMIN`, the policy
   exempts it.
2. **The policy may be detached.** `CREATE OR REPLACE DYNAMIC TABLE` silently
   detaches row access policies and drops both normal and CALLER grants. If you
   re-ran notebook 01, re-run notebook 04 from `attach_policy` and then cell 3
   above.
3. **Cache.** Every viewer shares one container instance, and `st.cache_data` is
   global to it. These apps include `CURRENT_ROLE()` in the cache key for exactly
   this reason — if you edit them, keep it.

## 7 · Changing the app later

`FROM` copies the files **once**. A later `git push` does not update a deployed
app.

Do not fix that with `CREATE OR REPLACE` — it drops the live version and every
grant, including the caller grants, and the app then fails for everyone but you.
Add a version instead:

In [ ]:
ALTER GIT REPOSITORY DEFAQTO_DB.PUBLIC.WORKSHOP_REPO FETCH;

ALTER STREAMLIT DEFAQTO_PARTNER_INSIGHTS
  ADD VERSION FROM '@DEFAQTO_DB.PUBLIC.WORKSHOP_REPO/branches/main/streamlit/partner_insights/';

ALTER STREAMLIT DEFAQTO_PARTNER_INSIGHTS ADD LIVE VERSION FROM LAST;

-- All viewers share one container, so the new code lands when it restarts. If
-- the old behaviour persists, suspend and resume the compute pool.

## 8 · Removing just the apps

Frees the compute pool and leaves everything else you built alone.

In [ ]:
DROP STREAMLIT IF EXISTS DEFAQTO_INTERNAL_MI;
DROP STREAMLIT IF EXISTS DEFAQTO_PARTNER_INSIGHTS;

SHOW STREAMLITS IN SCHEMA IDENTIFIER($my_schema);